# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * Modify `__getitem__` implemented above to support slicing.
        

3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [1]:
from numbers import Number

class matrix:

    def __init__(self, *args):
        if len(args) == 2:
            n, m = args
            if not all((isinstance(v, int) and (not isinstance(v, bool)) for v in (n, m))):
                raise TypeError('n and m must be integers')
            if n <= 0 or m <= 0:
                raise ValueError('n and m must be positive')
            self.data = [[0] * m for _ in range(n)]
        elif len(args) == 1:
            self.data = self._validate(args[0])
        else:
            raise TypeError('matrix() takes (n, m) or a list of lists')

    @staticmethod
    def _validate(rows):
        if isinstance(rows, matrix):
            rows = rows.data
        if not isinstance(rows, list) or len(rows) == 0:
            raise ValueError('expected a non-empty list of lists')
        if not all((isinstance(r, list) for r in rows)):
            raise ValueError('every row must be a list')
        width = len(rows[0])
        if width == 0:
            raise ValueError('rows must not be empty')
        for k, r in enumerate(rows):
            if len(r) != width:
                raise ValueError(f'row {k} has {len(r)} columns, expected {width}')
            for v in r:
                if isinstance(v, bool) or not isinstance(v, Number):
                    raise TypeError(f'non-numeric entry {v!r} in row {k}')
        return [list(r) for r in rows]

    @property
    def shape(self):
        return (len(self.data), len(self.data[0]))

    def __getitem__(self, key):
        if isinstance(key, tuple):
            i, j = key
            return self.data[i][j]
        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        elif isinstance(key, slice) and key == slice(None):
            self.assign(value)
        else:
            if not isinstance(value, list) or len(value) != self.shape[1]:
                raise ValueError('row has wrong length')
            self.data[key] = list(value)

    def assign(self, other):
        new = self._validate(other)
        if (len(new), len(new[0])) != self.shape:
            raise ValueError(f'size mismatch: {self.shape} vs {(len(new), len(new[0]))}')
        self.data = new
        return self

    def __eq__(self, other):
        if isinstance(other, matrix):
            return self.data == other.data
        return NotImplemented

    def __repr__(self):
        return 'matrix(' + repr(self.data) + ')'

def raises(exc, f, *a):
    try:
        f(*a)
    except exc:
        return True
    except Exception as e:
        print('  wrong exception:', type(e).__name__, e)
        return False
    return False

def run_tests():
    Z = matrix(2, 3)
    assert Z.data == [[0, 0, 0], [0, 0, 0]] and Z.shape == (2, 3)
    assert raises(ValueError, matrix, 0, 3)
    assert raises(ValueError, matrix, -1, 3)
    assert raises(TypeError, matrix, 2.5, 3)
    print('PASS: matrix(n, m) gives zeros; bad sizes rejected')
    Z[0][0] = 7
    assert Z[1][0] == 0
    print('PASS: rows of a zero matrix are independent objects')
    A = matrix([[1, 2, 3], [4, 5, 6]])
    assert A.shape == (2, 3)
    assert raises(ValueError, matrix, [[1, 2, 3], [4, 5]])
    assert raises(ValueError, matrix, [[1, 2], [3, 4], [5]])
    assert raises(ValueError, matrix, [])
    assert raises(ValueError, matrix, [[]])
    assert raises(ValueError, matrix, [1, 2, 3])
    assert raises(TypeError, matrix, [[1, 'a'], [3, 4]])
    print('PASS: list-of-lists init validated (ragged/empty/non-numeric rejected)')
    src = [[1, 2], [3, 4]]
    B = matrix(src)
    src[0][0] = 99
    assert B[0][0] == 1
    print("PASS: matrix does not alias the caller's list")
    assert A[0][1] == 2 and A[0, 1] == 2
    assert A[1][2] == 6 and A[1, 2] == 6
    A[0][1] = 20
    assert A[0, 1] == 20
    A[1, 0] = 40
    assert A[1][0] == 40
    assert raises(IndexError, lambda: A[5, 0])
    assert raises(IndexError, lambda: A[0, 9])
    print('PASS: M[i][j] and M[i,j] work for reading and writing')
    M1 = matrix(2, 2)
    M2 = matrix([[1, 2], [3, 4]])
    M1.assign(M2)
    assert M1 == M2 and M1 is not M2
    M2[0, 0] = 100
    assert M1[0, 0] == 1
    print('PASS: M1.assign(M2) copies values (matrix, same size)')
    M1[:] = matrix([[5, 6], [7, 8]])
    assert M1.data == [[5, 6], [7, 8]]
    print('PASS: M1[:] = M2 copies values')
    M1.assign([[9, 10], [11, 12]])
    assert M1.data == [[9, 10], [11, 12]]
    M1[:] = [[0, 1], [2, 3]]
    assert M1.data == [[0, 1], [2, 3]]
    print('PASS: assignment from list of lists of correct size')
    assert raises(ValueError, M1.assign, matrix(3, 3))
    assert raises(ValueError, M1.assign, [[1, 2, 3], [4, 5, 6]])
    assert raises(ValueError, M1.assign, [[1, 2], [3]])
    assert M1.data == [[0, 1], [2, 3]]
    print('PASS: size mismatch / malformed input raises error, leaves M1 unchanged')
    print('\nAll tests passed.')
if __name__ == '__main__':
    run_tests()

PASS: matrix(n, m) gives zeros; bad sizes rejected
PASS: rows of a zero matrix are independent objects
PASS: list-of-lists init validated (ragged/empty/non-numeric rejected)
PASS: matrix does not alias the caller's list
PASS: M[i][j] and M[i,j] work for reading and writing
PASS: M1.assign(M2) copies values (matrix, same size)
PASS: M1[:] = M2 copies values
PASS: assignment from list of lists of correct size
PASS: size mismatch / malformed input raises error, leaves M1 unchanged

All tests passed.


In [2]:
from numbers import Number

class matrix:
    def __init__(self, *args):
        if len(args) == 2:
            n, m = args
            if not all(isinstance(v, int) and not isinstance(v, bool) for v in (n, m)):
                raise TypeError("n and m must be integers")
            if n <= 0 or m <= 0:
                raise ValueError("n and m must be positive")
            self.data = [[0] * m for _ in range(n)]
        elif len(args) == 1:
            self.data = self._validate(args[0])
        else:
            raise TypeError("matrix() takes (n, m) or a list of lists")

    @staticmethod
    def _validate(rows):
        if isinstance(rows, matrix):
            rows = rows.data
        if not isinstance(rows, list) or len(rows) == 0:
            raise ValueError("expected a non-empty list of lists")
        if not all(isinstance(r, list) for r in rows):
            raise ValueError("every row must be a list")
        width = len(rows[0])
        if width == 0:
            raise ValueError("rows must not be empty")
        for k, r in enumerate(rows):
            if len(r) != width:
                raise ValueError(f"row {k} has {len(r)} columns, expected {width}")
            for v in r:
                if isinstance(v, bool) or not isinstance(v, Number):
                    raise TypeError(f"non-numeric entry {v!r} in row {k}")
        return [list(r) for r in rows]

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        return matrix([list(col) for col in zip(*self.data)])

    def row(self, n):
        return matrix([list(self.data[n])])

    def column(self, n):
        return matrix([[r[n]] for r in self.data])

    def to_list(self):
        return [list(r) for r in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        n, m = self.shape()
        if not (0 <= n_0 < n_1 <= n and 0 <= m_0 < m_1 <= m):
            raise ValueError(
                f"block bounds ({n_0},{n_1},{m_0},{m_1}) invalid for shape {(n, m)}"
            )
        return matrix([r[m_0:m_1] for r in self.data[n_0:n_1]])

    def __getitem__(self, key):
        if isinstance(key, tuple):
            if len(key) != 2:
                raise IndexError("matrix indices must be M[i], M[i,j] or slices thereof")
            i, j = key
            if not isinstance(i, slice) and not isinstance(j, slice):
                return self.data[i][j]
            rows = self.data[i] if isinstance(i, slice) else [self.data[i]]
            if isinstance(j, slice):
                return matrix([r[j] for r in rows])
            return matrix([[r[j]] for r in rows])
        if isinstance(key, slice):
            return matrix(self.data[key])
        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        elif isinstance(key, slice) and key == slice(None):
            self.assign(value)
        else:
            if not isinstance(value, list) or len(value) != self.shape()[1]:
                raise ValueError("row has wrong length")
            self.data[key] = list(value)

    def assign(self, other):
        new = self._validate(other)
        if (len(new), len(new[0])) != self.shape():
            raise ValueError(f"size mismatch: {self.shape()} vs {(len(new), len(new[0]))}")
        self.data = new
        return self

    def __eq__(self, other):
        if isinstance(other, matrix):
            return self.data == other.data
        return NotImplemented

    def __repr__(self):
        return "matrix(" + repr(self.data) + ")"


def raises(exc, f, *a):
    try:
        f(*a)
    except exc:
        return True
    except Exception as e:
        print("  wrong exception:", type(e).__name__, e)
        return False
    return False


def test_part1():
    Z = matrix(2, 3)
    assert Z.data == [[0, 0, 0], [0, 0, 0]] and Z.shape() == (2, 3)
    assert raises(ValueError, matrix, 0, 3)
    assert raises(ValueError, matrix, -1, 3)
    assert raises(TypeError, matrix, 2.5, 3)
    Z[0][0] = 7
    assert Z[1][0] == 0
    print("PASS: matrix(n, m) gives independent rows of zeros; bad sizes rejected")

    A = matrix([[1, 2, 3], [4, 5, 6]])
    assert A.shape() == (2, 3)
    assert raises(ValueError, matrix, [[1, 2, 3], [4, 5]])
    assert raises(ValueError, matrix, [[1, 2], [3, 4], [5]])
    assert raises(ValueError, matrix, [])
    assert raises(ValueError, matrix, [[]])
    assert raises(ValueError, matrix, [1, 2, 3])
    assert raises(TypeError, matrix, [[1, "a"], [3, 4]])
    src = [[1, 2], [3, 4]]
    B = matrix(src)
    src[0][0] = 99
    assert B[0][0] == 1
    print("PASS: list-of-lists init validated and copied")

    assert A[0][1] == 2 and A[0, 1] == 2
    A[0][1] = 20
    assert A[0, 1] == 20
    A[1, 0] = 40
    assert A[1][0] == 40
    assert raises(IndexError, lambda: A[5, 0])
    assert raises(IndexError, lambda: A[0, 9])
    print("PASS: M[i][j] and M[i,j] read and write")

    M1 = matrix(2, 2)
    M2 = matrix([[1, 2], [3, 4]])
    M1.assign(M2)
    assert M1 == M2 and M1 is not M2
    M2[0, 0] = 100
    assert M1[0, 0] == 1
    M1[:] = matrix([[5, 6], [7, 8]])
    assert M1.data == [[5, 6], [7, 8]]
    M1.assign([[9, 10], [11, 12]])
    M1[:] = [[0, 1], [2, 3]]
    assert M1.data == [[0, 1], [2, 3]]
    assert raises(ValueError, M1.assign, matrix(3, 3))
    assert raises(ValueError, M1.assign, [[1, 2, 3], [4, 5, 6]])
    assert raises(ValueError, M1.assign, [[1, 2], [3]])
    assert M1.data == [[0, 1], [2, 3]]
    print("PASS: assignment from matrix / list of lists, errors on bad size")


def test_part2():
    A = matrix([[1, 2, 3], [4, 5, 6]])

    s = A.shape()
    assert s == (2, 3) and isinstance(s, tuple)
    assert matrix(5, 1).shape() == (5, 1)
    print("PASS: shape() returns (n, m) tuple")

    T = A.transpose()
    assert T == matrix([[1, 4], [2, 5], [3, 6]])
    assert T.shape() == (3, 2)
    assert T is not A
    assert T.transpose() == A
    T[0, 0] = 99
    assert A[0, 0] == 1
    assert matrix([[1, 2], [3, 4]]).transpose() == matrix([[1, 3], [2, 4]])
    assert matrix([[1, 2, 3]]).transpose().shape() == (3, 1)
    print("PASS: transpose() correct, new instance, double transpose is identity")

    r = A.row(0)
    assert r == matrix([[1, 2, 3]]) and r.shape() == (1, 3)
    assert A.row(1) == matrix([[4, 5, 6]])
    c = A.column(1)
    assert c == matrix([[2], [5]]) and c.shape() == (2, 1)
    assert A.column(2) == matrix([[3], [6]])
    r[0, 0] = -1
    c[0, 0] = -1
    assert A[0, 0] == 1 and A[0, 1] == 2
    assert raises(IndexError, A.row, 2)
    assert raises(IndexError, A.column, 3)
    print("PASS: row(n)/column(n) return correctly shaped independent matrices")

    L = A.to_list()
    assert L == [[1, 2, 3], [4, 5, 6]] and isinstance(L, list)
    L[0][0] = 99
    assert A[0, 0] == 1
    print("PASS: to_list() returns a list of lists copy")

    B = matrix([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])
    blk = B.block(0, 2, 1, 3)
    assert blk == matrix([[2, 3], [6, 7]]) and blk.shape() == (2, 2)
    assert B.block(0, 3, 0, 4) == B
    assert B.block(2, 3, 3, 4) == matrix([[12]])
    blk[0, 0] = -1
    assert B[0, 1] == 2
    assert raises(ValueError, B.block, 2, 2, 0, 1)
    assert raises(ValueError, B.block, 0, 4, 0, 1)
    assert raises(ValueError, B.block, 0, 1, 0, 5)
    assert raises(ValueError, B.block, -1, 1, 0, 1)
    assert raises(ValueError, B.block, 2, 1, 0, 1)
    print("PASS: block(n_0,n_1,m_0,m_1) returns the right submatrix, bounds checked")

    assert B[1, 2] == 7 and B[1][2] == 7
    assert B[0:2] == matrix([[1, 2, 3, 4], [5, 6, 7, 8]])
    assert B[::2] == matrix([[1, 2, 3, 4], [9, 10, 11, 12]])
    assert B[1:, 2:] == matrix([[7, 8], [11, 12]])
    assert B[0:2, 1:3] == B.block(0, 2, 1, 3)
    assert B[:, 1] == matrix([[2], [6], [10]]) and B[:, 1].shape() == (3, 1)
    assert B[1, :] == matrix([[5, 6, 7, 8]]) and B[1, :].shape() == (1, 4)
    assert B[:, :] == B and B[:, :] is not B
    assert B[-1, -2:] == matrix([[11, 12]])
    sl = B[0:2, 0:2]
    sl[0, 0] = -1
    assert B[0, 0] == 1
    assert raises(ValueError, lambda: B[1:1])
    assert raises(ValueError, lambda: B[0:2, 5:])
    assert raises(IndexError, lambda: B[9, 0:2])
    assert raises(IndexError, lambda: B[0, 1, 2])
    print("PASS: __getitem__ supports slices (rows, submatrices, row/column forms), copies")

    print("\nAll tests passed.")


if __name__ == "__main__":
    test_part1()
    test_part2()

PASS: matrix(n, m) gives independent rows of zeros; bad sizes rejected
PASS: list-of-lists init validated and copied
PASS: M[i][j] and M[i,j] read and write
PASS: assignment from matrix / list of lists, errors on bad size
PASS: shape() returns (n, m) tuple
PASS: transpose() correct, new instance, double transpose is identity
PASS: row(n)/column(n) return correctly shaped independent matrices
PASS: to_list() returns a list of lists copy
PASS: block(n_0,n_1,m_0,m_1) returns the right submatrix, bounds checked
PASS: __getitem__ supports slices (rows, submatrices, row/column forms), copies

All tests passed.


In [3]:
from numbers import Number

class matrix:
    def __init__(self, *args):
        if len(args) == 2:
            n, m = args
            if not all(isinstance(v, int) and not isinstance(v, bool) for v in (n, m)):
                raise TypeError("n and m must be integers")
            if n <= 0 or m <= 0:
                raise ValueError("n and m must be positive")
            self.data = [[0] * m for _ in range(n)]
        elif len(args) == 1:
            self.data = self._validate(args[0])
        else:
            raise TypeError("matrix() takes (n, m) or a list of lists")

    @staticmethod
    def _validate(rows):
        if isinstance(rows, matrix):
            rows = rows.data
        if not isinstance(rows, list) or len(rows) == 0:
            raise ValueError("expected a non-empty list of lists")
        if not all(isinstance(r, list) for r in rows):
            raise ValueError("every row must be a list")
        width = len(rows[0])
        if width == 0:
            raise ValueError("rows must not be empty")
        for k, r in enumerate(rows):
            if len(r) != width:
                raise ValueError(f"row {k} has {len(r)} columns, expected {width}")
            for v in r:
                if isinstance(v, bool) or not isinstance(v, Number):
                    raise TypeError(f"non-numeric entry {v!r} in row {k}")
        return [list(r) for r in rows]

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        return matrix([list(col) for col in zip(*self.data)])

    def row(self, n):
        return matrix([list(self.data[n])])

    def column(self, n):
        return matrix([[r[n]] for r in self.data])

    def to_list(self):
        return [list(r) for r in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        n, m = self.shape()
        if not (0 <= n_0 < n_1 <= n and 0 <= m_0 < m_1 <= m):
            raise ValueError(
                f"block bounds ({n_0},{n_1},{m_0},{m_1}) invalid for shape {(n, m)}"
            )
        return matrix([r[m_0:m_1] for r in self.data[n_0:n_1]])

    def __getitem__(self, key):
        if isinstance(key, tuple):
            if len(key) != 2:
                raise IndexError("matrix indices must be M[i], M[i,j] or slices thereof")
            i, j = key
            if not isinstance(i, slice) and not isinstance(j, slice):
                return self.data[i][j]
            rows = self.data[i] if isinstance(i, slice) else [self.data[i]]
            if isinstance(j, slice):
                return matrix([r[j] for r in rows])
            return matrix([[r[j]] for r in rows])
        if isinstance(key, slice):
            return matrix(self.data[key])
        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        elif isinstance(key, slice) and key == slice(None):
            self.assign(value)
        else:
            if not isinstance(value, list) or len(value) != self.shape()[1]:
                raise ValueError("row has wrong length")
            self.data[key] = list(value)

    def assign(self, other):
        new = self._validate(other)
        if (len(new), len(new[0])) != self.shape():
            raise ValueError(f"size mismatch: {self.shape()} vs {(len(new), len(new[0]))}")
        self.data = new
        return self

    def __eq__(self, other):
        if isinstance(other, matrix):
            return self.data == other.data
        return NotImplemented

    def __repr__(self):
        return "matrix(" + repr(self.data) + ")"


def constant(n, m, c):
    if isinstance(c, bool) or not isinstance(c, Number):
        raise TypeError("c must be a real number")
    M = matrix(n, m)
    M.data = [[float(c)] * m for _ in range(n)]
    return M


def zeros(n, m):
    return constant(n, m, 0)


def ones(n, m):
    return constant(n, m, 1)


def eye(n):
    M = matrix(n, n)
    M.data = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    return M


def raises(exc, f, *a):
    try:
        f(*a)
    except exc:
        return True
    except Exception as e:
        print("  wrong exception:", type(e).__name__, e)
        return False
    return False


def test_part1():
    Z = matrix(2, 3)
    assert Z.data == [[0, 0, 0], [0, 0, 0]] and Z.shape() == (2, 3)
    assert raises(ValueError, matrix, 0, 3)
    assert raises(ValueError, matrix, -1, 3)
    assert raises(TypeError, matrix, 2.5, 3)
    Z[0][0] = 7
    assert Z[1][0] == 0
    print("PASS: matrix(n, m) gives independent rows of zeros; bad sizes rejected")

    A = matrix([[1, 2, 3], [4, 5, 6]])
    assert A.shape() == (2, 3)
    assert raises(ValueError, matrix, [[1, 2, 3], [4, 5]])
    assert raises(ValueError, matrix, [[1, 2], [3, 4], [5]])
    assert raises(ValueError, matrix, [])
    assert raises(ValueError, matrix, [[]])
    assert raises(ValueError, matrix, [1, 2, 3])
    assert raises(TypeError, matrix, [[1, "a"], [3, 4]])
    src = [[1, 2], [3, 4]]
    B = matrix(src)
    src[0][0] = 99
    assert B[0][0] == 1
    print("PASS: list-of-lists init validated and copied")

    assert A[0][1] == 2 and A[0, 1] == 2
    A[0][1] = 20
    assert A[0, 1] == 20
    A[1, 0] = 40
    assert A[1][0] == 40
    assert raises(IndexError, lambda: A[5, 0])
    assert raises(IndexError, lambda: A[0, 9])
    print("PASS: M[i][j] and M[i,j] read and write")

    M1 = matrix(2, 2)
    M2 = matrix([[1, 2], [3, 4]])
    M1.assign(M2)
    assert M1 == M2 and M1 is not M2
    M2[0, 0] = 100
    assert M1[0, 0] == 1
    M1[:] = matrix([[5, 6], [7, 8]])
    assert M1.data == [[5, 6], [7, 8]]
    M1.assign([[9, 10], [11, 12]])
    M1[:] = [[0, 1], [2, 3]]
    assert M1.data == [[0, 1], [2, 3]]
    assert raises(ValueError, M1.assign, matrix(3, 3))
    assert raises(ValueError, M1.assign, [[1, 2, 3], [4, 5, 6]])
    assert raises(ValueError, M1.assign, [[1, 2], [3]])
    assert M1.data == [[0, 1], [2, 3]]
    print("PASS: assignment from matrix / list of lists, errors on bad size")


def test_part2():
    A = matrix([[1, 2, 3], [4, 5, 6]])

    s = A.shape()
    assert s == (2, 3) and isinstance(s, tuple)
    assert matrix(5, 1).shape() == (5, 1)
    print("PASS: shape() returns (n, m) tuple")

    T = A.transpose()
    assert T == matrix([[1, 4], [2, 5], [3, 6]])
    assert T.shape() == (3, 2)
    assert T is not A
    assert T.transpose() == A
    T[0, 0] = 99
    assert A[0, 0] == 1
    assert matrix([[1, 2], [3, 4]]).transpose() == matrix([[1, 3], [2, 4]])
    assert matrix([[1, 2, 3]]).transpose().shape() == (3, 1)
    print("PASS: transpose() correct, new instance, double transpose is identity")

    r = A.row(0)
    assert r == matrix([[1, 2, 3]]) and r.shape() == (1, 3)
    assert A.row(1) == matrix([[4, 5, 6]])
    c = A.column(1)
    assert c == matrix([[2], [5]]) and c.shape() == (2, 1)
    assert A.column(2) == matrix([[3], [6]])
    r[0, 0] = -1
    c[0, 0] = -1
    assert A[0, 0] == 1 and A[0, 1] == 2
    assert raises(IndexError, A.row, 2)
    assert raises(IndexError, A.column, 3)
    print("PASS: row(n)/column(n) return correctly shaped independent matrices")

    L = A.to_list()
    assert L == [[1, 2, 3], [4, 5, 6]] and isinstance(L, list)
    L[0][0] = 99
    assert A[0, 0] == 1
    print("PASS: to_list() returns a list of lists copy")

    B = matrix([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])
    blk = B.block(0, 2, 1, 3)
    assert blk == matrix([[2, 3], [6, 7]]) and blk.shape() == (2, 2)
    assert B.block(0, 3, 0, 4) == B
    assert B.block(2, 3, 3, 4) == matrix([[12]])
    blk[0, 0] = -1
    assert B[0, 1] == 2
    assert raises(ValueError, B.block, 2, 2, 0, 1)
    assert raises(ValueError, B.block, 0, 4, 0, 1)
    assert raises(ValueError, B.block, 0, 1, 0, 5)
    assert raises(ValueError, B.block, -1, 1, 0, 1)
    assert raises(ValueError, B.block, 2, 1, 0, 1)
    print("PASS: block(n_0,n_1,m_0,m_1) returns the right submatrix, bounds checked")

    assert B[1, 2] == 7 and B[1][2] == 7
    assert B[0:2] == matrix([[1, 2, 3, 4], [5, 6, 7, 8]])
    assert B[::2] == matrix([[1, 2, 3, 4], [9, 10, 11, 12]])
    assert B[1:, 2:] == matrix([[7, 8], [11, 12]])
    assert B[0:2, 1:3] == B.block(0, 2, 1, 3)
    assert B[:, 1] == matrix([[2], [6], [10]]) and B[:, 1].shape() == (3, 1)
    assert B[1, :] == matrix([[5, 6, 7, 8]]) and B[1, :].shape() == (1, 4)
    assert B[:, :] == B and B[:, :] is not B
    assert B[-1, -2:] == matrix([[11, 12]])
    sl = B[0:2, 0:2]
    sl[0, 0] = -1
    assert B[0, 0] == 1
    assert raises(ValueError, lambda: B[1:1])
    assert raises(ValueError, lambda: B[0:2, 5:])
    assert raises(IndexError, lambda: B[9, 0:2])
    assert raises(IndexError, lambda: B[0, 1, 2])
    print("PASS: __getitem__ supports slices (rows, submatrices, row/column forms), copies")

    print("\nAll tests passed.")


def all_floats(M):
    return all(type(v) is float for r in M.data for v in r)


def test_part3():
    C = constant(2, 3, 5)
    assert C.shape() == (2, 3)
    assert C.data == [[5.0, 5.0, 5.0], [5.0, 5.0, 5.0]]
    assert all_floats(C)
    assert constant(1, 4, -2.5).data == [[-2.5, -2.5, -2.5, -2.5]]
    assert constant(3, 1, 0).shape() == (3, 1)
    C[0, 0] = 9.0
    assert C[1, 0] == 5.0
    assert raises(ValueError, constant, 0, 3, 1)
    assert raises(TypeError, constant, 2.0, 3, 1)
    assert raises(TypeError, constant, 2, 3, "a")
    assert raises(TypeError, constant, 2, 3, True)
    print("PASS: constant(n,m,c) gives n x m floats of value c, rows independent, bad input rejected")

    Z = zeros(3, 2)
    assert Z.shape() == (3, 2) and Z.data == [[0.0, 0.0]] * 3
    assert all_floats(Z)
    O = ones(2, 4)
    assert O.shape() == (2, 4) and O.data == [[1.0] * 4] * 2
    assert all_floats(O)
    assert zeros(1, 1) == matrix([[0.0]]) and ones(1, 1) == matrix([[1.0]])
    assert raises(ValueError, zeros, -1, 2)
    assert raises(ValueError, ones, 2, 0)
    print("PASS: zeros(n,m) and ones(n,m) give float matrices of 0.0 and 1.0")

    I = eye(3)
    assert I.shape() == (3, 3)
    assert I.data == [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
    assert all_floats(I)
    assert eye(1) == matrix([[1.0]])
    assert I.transpose() == I
    assert all(I[i, j] == (1.0 if i == j else 0.0) for i in range(3) for j in range(3))
    big = eye(6)
    assert sum(big[i, i] for i in range(6)) == 6.0 and sum(map(sum, big.data)) == 6.0
    I[0, 0] = 5.0
    assert eye(3)[0, 0] == 1.0
    assert raises(ValueError, eye, 0)
    assert raises(TypeError, eye, 2.5)
    print("PASS: eye(n) is the n x n float identity, new object each call")

    print("\nAll part 3 tests passed.")


if __name__ == "__main__":
    test_part1()
    test_part2()
    test_part3()

PASS: matrix(n, m) gives independent rows of zeros; bad sizes rejected
PASS: list-of-lists init validated and copied
PASS: M[i][j] and M[i,j] read and write
PASS: assignment from matrix / list of lists, errors on bad size
PASS: shape() returns (n, m) tuple
PASS: transpose() correct, new instance, double transpose is identity
PASS: row(n)/column(n) return correctly shaped independent matrices
PASS: to_list() returns a list of lists copy
PASS: block(n_0,n_1,m_0,m_1) returns the right submatrix, bounds checked
PASS: __getitem__ supports slices (rows, submatrices, row/column forms), copies

All tests passed.
PASS: constant(n,m,c) gives n x m floats of value c, rows independent, bad input rejected
PASS: zeros(n,m) and ones(n,m) give float matrices of 0.0 and 1.0
PASS: eye(n) is the n x n float identity, new object each call

All part 3 tests passed.


In [4]:
from numbers import Number

class matrix:
    def __init__(self, *args):
        if len(args) == 2:
            n, m = args
            if not all(isinstance(v, int) and not isinstance(v, bool) for v in (n, m)):
                raise TypeError("n and m must be integers")
            if n <= 0 or m <= 0:
                raise ValueError("n and m must be positive")
            self.data = [[0] * m for _ in range(n)]
        elif len(args) == 1:
            self.data = self._validate(args[0])
        else:
            raise TypeError("matrix() takes (n, m) or a list of lists")

    @staticmethod
    def _validate(rows):
        if isinstance(rows, matrix):
            rows = rows.data
        if not isinstance(rows, list) or len(rows) == 0:
            raise ValueError("expected a non-empty list of lists")
        if not all(isinstance(r, list) for r in rows):
            raise ValueError("every row must be a list")
        width = len(rows[0])
        if width == 0:
            raise ValueError("rows must not be empty")
        for k, r in enumerate(rows):
            if len(r) != width:
                raise ValueError(f"row {k} has {len(r)} columns, expected {width}")
            for v in r:
                if isinstance(v, bool) or not isinstance(v, Number):
                    raise TypeError(f"non-numeric entry {v!r} in row {k}")
        return [list(r) for r in rows]

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        return matrix([list(col) for col in zip(*self.data)])

    def row(self, n):
        return matrix([list(self.data[n])])

    def column(self, n):
        return matrix([[r[n]] for r in self.data])

    def to_list(self):
        return [list(r) for r in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        n, m = self.shape()
        if not (0 <= n_0 < n_1 <= n and 0 <= m_0 < m_1 <= m):
            raise ValueError(
                f"block bounds ({n_0},{n_1},{m_0},{m_1}) invalid for shape {(n, m)}"
            )
        return matrix([r[m_0:m_1] for r in self.data[n_0:n_1]])

    @staticmethod
    def _require_matrix(N):
        if not isinstance(N, matrix):
            raise TypeError(f"expected a matrix, got {type(N).__name__}")

    def _require_same_shape(self, N, op):
        self._require_matrix(N)
        if self.shape() != N.shape():
            raise ValueError(f"{op}: shape mismatch {self.shape()} vs {N.shape()}")

    def scalarmul(self, c):
        if isinstance(c, bool) or not isinstance(c, Number):
            raise TypeError("c must be a number")
        return matrix([[c * v for v in r] for r in self.data])

    def add(self, N):
        self._require_same_shape(N, "add")
        return matrix([[a + b for a, b in zip(r, s)] for r, s in zip(self.data, N.data)])

    def sub(self, N):
        self._require_same_shape(N, "sub")
        return matrix([[a - b for a, b in zip(r, s)] for r, s in zip(self.data, N.data)])

    def mat_mult(self, N):
        self._require_matrix(N)
        if self.shape()[1] != N.shape()[0]:
            raise ValueError(
                f"mat_mult: inner dimensions differ, {self.shape()} times {N.shape()}"
            )
        cols = list(zip(*N.data))
        return matrix([[sum(a * b for a, b in zip(r, col)) for col in cols] for r in self.data])

    def element_mult(self, N):
        self._require_same_shape(N, "element_mult")
        return matrix([[a * b for a, b in zip(r, s)] for r, s in zip(self.data, N.data)])

    def equals(self, N):
        return isinstance(N, matrix) and self.data == N.data

    def __getitem__(self, key):
        if isinstance(key, tuple):
            if len(key) != 2:
                raise IndexError("matrix indices must be M[i], M[i,j] or slices thereof")
            i, j = key
            if not isinstance(i, slice) and not isinstance(j, slice):
                return self.data[i][j]
            rows = self.data[i] if isinstance(i, slice) else [self.data[i]]
            if isinstance(j, slice):
                return matrix([r[j] for r in rows])
            return matrix([[r[j]] for r in rows])
        if isinstance(key, slice):
            return matrix(self.data[key])
        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        elif isinstance(key, slice) and key == slice(None):
            self.assign(value)
        else:
            if not isinstance(value, list) or len(value) != self.shape()[1]:
                raise ValueError("row has wrong length")
            self.data[key] = list(value)

    def assign(self, other):
        new = self._validate(other)
        if (len(new), len(new[0])) != self.shape():
            raise ValueError(f"size mismatch: {self.shape()} vs {(len(new), len(new[0]))}")
        self.data = new
        return self

    def __eq__(self, other):
        if isinstance(other, matrix):
            return self.data == other.data
        return NotImplemented

    def __repr__(self):
        return "matrix(" + repr(self.data) + ")"


def constant(n, m, c):
    if isinstance(c, bool) or not isinstance(c, Number):
        raise TypeError("c must be a real number")
    M = matrix(n, m)
    M.data = [[float(c)] * m for _ in range(n)]
    return M


def zeros(n, m):
    return constant(n, m, 0)


def ones(n, m):
    return constant(n, m, 1)


def eye(n):
    M = matrix(n, n)
    M.data = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    return M


def raises(exc, f, *a):
    try:
        f(*a)
    except exc:
        return True
    except Exception as e:
        print("  wrong exception:", type(e).__name__, e)
        return False
    return False


def test_part1():
    Z = matrix(2, 3)
    assert Z.data == [[0, 0, 0], [0, 0, 0]] and Z.shape() == (2, 3)
    assert raises(ValueError, matrix, 0, 3)
    assert raises(ValueError, matrix, -1, 3)
    assert raises(TypeError, matrix, 2.5, 3)
    Z[0][0] = 7
    assert Z[1][0] == 0
    print("PASS: matrix(n, m) gives independent rows of zeros; bad sizes rejected")

    A = matrix([[1, 2, 3], [4, 5, 6]])
    assert A.shape() == (2, 3)
    assert raises(ValueError, matrix, [[1, 2, 3], [4, 5]])
    assert raises(ValueError, matrix, [[1, 2], [3, 4], [5]])
    assert raises(ValueError, matrix, [])
    assert raises(ValueError, matrix, [[]])
    assert raises(ValueError, matrix, [1, 2, 3])
    assert raises(TypeError, matrix, [[1, "a"], [3, 4]])
    src = [[1, 2], [3, 4]]
    B = matrix(src)
    src[0][0] = 99
    assert B[0][0] == 1
    print("PASS: list-of-lists init validated and copied")

    assert A[0][1] == 2 and A[0, 1] == 2
    A[0][1] = 20
    assert A[0, 1] == 20
    A[1, 0] = 40
    assert A[1][0] == 40
    assert raises(IndexError, lambda: A[5, 0])
    assert raises(IndexError, lambda: A[0, 9])
    print("PASS: M[i][j] and M[i,j] read and write")

    M1 = matrix(2, 2)
    M2 = matrix([[1, 2], [3, 4]])
    M1.assign(M2)
    assert M1 == M2 and M1 is not M2
    M2[0, 0] = 100
    assert M1[0, 0] == 1
    M1[:] = matrix([[5, 6], [7, 8]])
    assert M1.data == [[5, 6], [7, 8]]
    M1.assign([[9, 10], [11, 12]])
    M1[:] = [[0, 1], [2, 3]]
    assert M1.data == [[0, 1], [2, 3]]
    assert raises(ValueError, M1.assign, matrix(3, 3))
    assert raises(ValueError, M1.assign, [[1, 2, 3], [4, 5, 6]])
    assert raises(ValueError, M1.assign, [[1, 2], [3]])
    assert M1.data == [[0, 1], [2, 3]]
    print("PASS: assignment from matrix / list of lists, errors on bad size")


def test_part2():
    A = matrix([[1, 2, 3], [4, 5, 6]])

    s = A.shape()
    assert s == (2, 3) and isinstance(s, tuple)
    assert matrix(5, 1).shape() == (5, 1)
    print("PASS: shape() returns (n, m) tuple")

    T = A.transpose()
    assert T == matrix([[1, 4], [2, 5], [3, 6]])
    assert T.shape() == (3, 2)
    assert T is not A
    assert T.transpose() == A
    T[0, 0] = 99
    assert A[0, 0] == 1
    assert matrix([[1, 2], [3, 4]]).transpose() == matrix([[1, 3], [2, 4]])
    assert matrix([[1, 2, 3]]).transpose().shape() == (3, 1)
    print("PASS: transpose() correct, new instance, double transpose is identity")

    r = A.row(0)
    assert r == matrix([[1, 2, 3]]) and r.shape() == (1, 3)
    assert A.row(1) == matrix([[4, 5, 6]])
    c = A.column(1)
    assert c == matrix([[2], [5]]) and c.shape() == (2, 1)
    assert A.column(2) == matrix([[3], [6]])
    r[0, 0] = -1
    c[0, 0] = -1
    assert A[0, 0] == 1 and A[0, 1] == 2
    assert raises(IndexError, A.row, 2)
    assert raises(IndexError, A.column, 3)
    print("PASS: row(n)/column(n) return correctly shaped independent matrices")

    L = A.to_list()
    assert L == [[1, 2, 3], [4, 5, 6]] and isinstance(L, list)
    L[0][0] = 99
    assert A[0, 0] == 1
    print("PASS: to_list() returns a list of lists copy")

    B = matrix([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])
    blk = B.block(0, 2, 1, 3)
    assert blk == matrix([[2, 3], [6, 7]]) and blk.shape() == (2, 2)
    assert B.block(0, 3, 0, 4) == B
    assert B.block(2, 3, 3, 4) == matrix([[12]])
    blk[0, 0] = -1
    assert B[0, 1] == 2
    assert raises(ValueError, B.block, 2, 2, 0, 1)
    assert raises(ValueError, B.block, 0, 4, 0, 1)
    assert raises(ValueError, B.block, 0, 1, 0, 5)
    assert raises(ValueError, B.block, -1, 1, 0, 1)
    assert raises(ValueError, B.block, 2, 1, 0, 1)
    print("PASS: block(n_0,n_1,m_0,m_1) returns the right submatrix, bounds checked")

    assert B[1, 2] == 7 and B[1][2] == 7
    assert B[0:2] == matrix([[1, 2, 3, 4], [5, 6, 7, 8]])
    assert B[::2] == matrix([[1, 2, 3, 4], [9, 10, 11, 12]])
    assert B[1:, 2:] == matrix([[7, 8], [11, 12]])
    assert B[0:2, 1:3] == B.block(0, 2, 1, 3)
    assert B[:, 1] == matrix([[2], [6], [10]]) and B[:, 1].shape() == (3, 1)
    assert B[1, :] == matrix([[5, 6, 7, 8]]) and B[1, :].shape() == (1, 4)
    assert B[:, :] == B and B[:, :] is not B
    assert B[-1, -2:] == matrix([[11, 12]])
    sl = B[0:2, 0:2]
    sl[0, 0] = -1
    assert B[0, 0] == 1
    assert raises(ValueError, lambda: B[1:1])
    assert raises(ValueError, lambda: B[0:2, 5:])
    assert raises(IndexError, lambda: B[9, 0:2])
    assert raises(IndexError, lambda: B[0, 1, 2])
    print("PASS: __getitem__ supports slices (rows, submatrices, row/column forms), copies")

    print("\nAll tests passed.")


def all_floats(M):
    return all(type(v) is float for r in M.data for v in r)


def test_part3():
    C = constant(2, 3, 5)
    assert C.shape() == (2, 3)
    assert C.data == [[5.0, 5.0, 5.0], [5.0, 5.0, 5.0]]
    assert all_floats(C)
    assert constant(1, 4, -2.5).data == [[-2.5, -2.5, -2.5, -2.5]]
    assert constant(3, 1, 0).shape() == (3, 1)
    C[0, 0] = 9.0
    assert C[1, 0] == 5.0
    assert raises(ValueError, constant, 0, 3, 1)
    assert raises(TypeError, constant, 2.0, 3, 1)
    assert raises(TypeError, constant, 2, 3, "a")
    assert raises(TypeError, constant, 2, 3, True)
    print("PASS: constant(n,m,c) gives n x m floats of value c, rows independent, bad input rejected")

    Z = zeros(3, 2)
    assert Z.shape() == (3, 2) and Z.data == [[0.0, 0.0]] * 3
    assert all_floats(Z)
    O = ones(2, 4)
    assert O.shape() == (2, 4) and O.data == [[1.0] * 4] * 2
    assert all_floats(O)
    assert zeros(1, 1) == matrix([[0.0]]) and ones(1, 1) == matrix([[1.0]])
    assert raises(ValueError, zeros, -1, 2)
    assert raises(ValueError, ones, 2, 0)
    print("PASS: zeros(n,m) and ones(n,m) give float matrices of 0.0 and 1.0")

    I = eye(3)
    assert I.shape() == (3, 3)
    assert I.data == [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
    assert all_floats(I)
    assert eye(1) == matrix([[1.0]])
    assert I.transpose() == I
    assert all(I[i, j] == (1.0 if i == j else 0.0) for i in range(3) for j in range(3))
    big = eye(6)
    assert sum(big[i, i] for i in range(6)) == 6.0 and sum(map(sum, big.data)) == 6.0
    I[0, 0] = 5.0
    assert eye(3)[0, 0] == 1.0
    assert raises(ValueError, eye, 0)
    assert raises(TypeError, eye, 2.5)
    print("PASS: eye(n) is the n x n float identity, new object each call")

    print("\nAll part 3 tests passed.")


def test_part4():
    A = matrix([[1, 2, 3], [4, 5, 6]])
    B = matrix([[7, 8, 9], [10, 11, 12]])
    C = matrix([[1, 2], [3, 4], [5, 6]])

    S = A.scalarmul(2)
    assert S == matrix([[2, 4, 6], [8, 10, 12]]) and S is not A
    assert A.scalarmul(0.5).data == [[0.5, 1.0, 1.5], [2.0, 2.5, 3.0]]
    assert A.scalarmul(-1) == matrix([[-1, -2, -3], [-4, -5, -6]])
    assert A.scalarmul(0) == matrix([[0, 0, 0], [0, 0, 0]])
    assert A.scalarmul(1) == A
    assert A.scalarmul(2).shape() == (2, 3)
    assert A == matrix([[1, 2, 3], [4, 5, 6]])
    assert raises(TypeError, A.scalarmul, "a")
    assert raises(TypeError, A.scalarmul, [1])
    assert raises(TypeError, A.scalarmul, True)
    print("PASS: scalarmul(c) correct, new matrix, original unchanged, bad c rejected")

    assert A.add(B) == matrix([[8, 10, 12], [14, 16, 18]])
    assert A.add(B) == B.add(A)
    assert A.add(zeros(2, 3)) == A
    assert A.add(A) == A.scalarmul(2)
    assert A.add(B).shape() == (2, 3)
    assert raises(ValueError, A.add, C)
    assert raises(ValueError, A.add, matrix(2, 2))
    assert raises(ValueError, A.add, matrix(3, 2))
    assert raises(TypeError, A.add, [[1, 2, 3], [4, 5, 6]])
    assert A == matrix([[1, 2, 3], [4, 5, 6]]) and B == matrix([[7, 8, 9], [10, 11, 12]])
    print("PASS: add(N) correct, shapes checked, operands unchanged")

    assert A.sub(B) == matrix([[-6, -6, -6], [-6, -6, -6]])
    assert A.sub(A) == zeros(2, 3)
    assert A.sub(B).add(B) == A
    assert B.sub(A) == A.sub(B).scalarmul(-1)
    assert raises(ValueError, A.sub, C)
    assert raises(ValueError, A.sub, matrix(1, 3))
    assert raises(TypeError, A.sub, 5)
    print("PASS: sub(N) correct, shapes checked")

    P = A.mat_mult(C)
    assert P == matrix([[22, 28], [49, 64]]) and P.shape() == (2, 2)
    Q = C.mat_mult(A)
    assert Q == matrix([[9, 12, 15], [19, 26, 33], [29, 40, 51]]) and Q.shape() == (3, 3)
    assert P.shape() != Q.shape()
    assert A.mat_mult(eye(3)) == A
    assert eye(2).mat_mult(A) == A
    v = matrix([[1], [0], [1]])
    assert A.mat_mult(v) == matrix([[4], [10]]) and A.mat_mult(v).shape() == (2, 1)
    assert matrix([[1, 2, 3]]).mat_mult(v) == matrix([[4]])
    assert v.mat_mult(matrix([[1, 2]])).shape() == (3, 2)
    assert A.mat_mult(C).mat_mult(A) == A.mat_mult(C.mat_mult(A))
    assert A.mat_mult(C).transpose() == C.transpose().mat_mult(A.transpose())
    assert A.mat_mult(ones(3, 2).add(ones(3, 2))) == A.mat_mult(ones(3, 2)).scalarmul(2)
    assert raises(ValueError, A.mat_mult, B)
    assert raises(ValueError, A.mat_mult, matrix(2, 2))
    assert raises(ValueError, C.mat_mult, C)
    assert raises(TypeError, A.mat_mult, [[1], [2], [3]])
    print("PASS: mat_mult(N) correct shapes/values, identity and algebraic laws, inner dims checked")

    E = A.element_mult(B)
    assert E == matrix([[7, 16, 27], [40, 55, 72]]) and E.shape() == (2, 3)
    assert E == B.element_mult(A)
    assert A.element_mult(ones(2, 3)) == A
    assert A.element_mult(zeros(2, 3)) == zeros(2, 3)
    assert raises(ValueError, A.element_mult, C)
    assert raises(ValueError, A.element_mult, matrix(2, 4))
    assert raises(TypeError, A.element_mult, 2)
    print("PASS: element_mult(N) correct, shapes checked (2x3 vs 3x2 rejected)")

    assert A.equals(matrix([[1, 2, 3], [4, 5, 6]])) is True
    assert A.equals(A) is True
    assert A.equals(B) is False
    assert A.equals(matrix([[1, 2, 3], [4, 5, 7]])) is False
    assert A.equals(C) is False
    assert A.equals(A.transpose()) is False
    assert A.equals(matrix([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])) is True
    assert A.equals([[1, 2, 3], [4, 5, 6]]) is False
    assert A.equals(None) is False
    print("PASS: equals(N) returns True/False, different shapes are simply False")

    print("\nAll part 4 tests passed.")


if __name__ == "__main__":
    test_part1()
    test_part2()
    test_part3()
    test_part4()

PASS: matrix(n, m) gives independent rows of zeros; bad sizes rejected
PASS: list-of-lists init validated and copied
PASS: M[i][j] and M[i,j] read and write
PASS: assignment from matrix / list of lists, errors on bad size
PASS: shape() returns (n, m) tuple
PASS: transpose() correct, new instance, double transpose is identity
PASS: row(n)/column(n) return correctly shaped independent matrices
PASS: to_list() returns a list of lists copy
PASS: block(n_0,n_1,m_0,m_1) returns the right submatrix, bounds checked
PASS: __getitem__ supports slices (rows, submatrices, row/column forms), copies

All tests passed.
PASS: constant(n,m,c) gives n x m floats of value c, rows independent, bad input rejected
PASS: zeros(n,m) and ones(n,m) give float matrices of 0.0 and 1.0
PASS: eye(n) is the n x n float identity, new object each call

All part 3 tests passed.
PASS: scalarmul(c) correct, new matrix, original unchanged, bad c rejected
PASS: add(N) correct, shapes checked, operands unchanged
PASS: sub(N

In [5]:
from numbers import Number

class matrix:
    def __init__(self, *args):
        if len(args) == 2:
            n, m = args
            if not all(isinstance(v, int) and not isinstance(v, bool) for v in (n, m)):
                raise TypeError("n and m must be integers")
            if n <= 0 or m <= 0:
                raise ValueError("n and m must be positive")
            self.data = [[0] * m for _ in range(n)]
        elif len(args) == 1:
            self.data = self._validate(args[0])
        else:
            raise TypeError("matrix() takes (n, m) or a list of lists")

    @staticmethod
    def _validate(rows):
        if isinstance(rows, matrix):
            rows = rows.data
        if not isinstance(rows, list) or len(rows) == 0:
            raise ValueError("expected a non-empty list of lists")
        if not all(isinstance(r, list) for r in rows):
            raise ValueError("every row must be a list")
        width = len(rows[0])
        if width == 0:
            raise ValueError("rows must not be empty")
        for k, r in enumerate(rows):
            if len(r) != width:
                raise ValueError(f"row {k} has {len(r)} columns, expected {width}")
            for v in r:
                if isinstance(v, bool) or not isinstance(v, Number):
                    raise TypeError(f"non-numeric entry {v!r} in row {k}")
        return [list(r) for r in rows]

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        return matrix([list(col) for col in zip(*self.data)])

    def row(self, n):
        return matrix([list(self.data[n])])

    def column(self, n):
        return matrix([[r[n]] for r in self.data])

    def to_list(self):
        return [list(r) for r in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        n, m = self.shape()
        if not (0 <= n_0 < n_1 <= n and 0 <= m_0 < m_1 <= m):
            raise ValueError(
                f"block bounds ({n_0},{n_1},{m_0},{m_1}) invalid for shape {(n, m)}"
            )
        return matrix([r[m_0:m_1] for r in self.data[n_0:n_1]])

    @staticmethod
    def _require_matrix(N):
        if not isinstance(N, matrix):
            raise TypeError(f"expected a matrix, got {type(N).__name__}")

    def _require_same_shape(self, N, op):
        self._require_matrix(N)
        if self.shape() != N.shape():
            raise ValueError(f"{op}: shape mismatch {self.shape()} vs {N.shape()}")

    def scalarmul(self, c):
        if isinstance(c, bool) or not isinstance(c, Number):
            raise TypeError("c must be a number")
        return matrix([[c * v for v in r] for r in self.data])

    def add(self, N):
        self._require_same_shape(N, "add")
        return matrix([[a + b for a, b in zip(r, s)] for r, s in zip(self.data, N.data)])

    def sub(self, N):
        self._require_same_shape(N, "sub")
        return matrix([[a - b for a, b in zip(r, s)] for r, s in zip(self.data, N.data)])

    def mat_mult(self, N):
        self._require_matrix(N)
        if self.shape()[1] != N.shape()[0]:
            raise ValueError(
                f"mat_mult: inner dimensions differ, {self.shape()} times {N.shape()}"
            )
        cols = list(zip(*N.data))
        return matrix([[sum(a * b for a, b in zip(r, col)) for col in cols] for r in self.data])

    def element_mult(self, N):
        self._require_same_shape(N, "element_mult")
        return matrix([[a * b for a, b in zip(r, s)] for r, s in zip(self.data, N.data)])

    def equals(self, N):
        return isinstance(N, matrix) and self.data == N.data

    def __getitem__(self, key):
        if isinstance(key, tuple):
            if len(key) != 2:
                raise IndexError("matrix indices must be M[i], M[i,j] or slices thereof")
            i, j = key
            if not isinstance(i, slice) and not isinstance(j, slice):
                return self.data[i][j]
            rows = self.data[i] if isinstance(i, slice) else [self.data[i]]
            if isinstance(j, slice):
                return matrix([r[j] for r in rows])
            return matrix([[r[j]] for r in rows])
        if isinstance(key, slice):
            return matrix(self.data[key])
        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        elif isinstance(key, slice) and key == slice(None):
            self.assign(value)
        else:
            if not isinstance(value, list) or len(value) != self.shape()[1]:
                raise ValueError("row has wrong length")
            self.data[key] = list(value)

    def assign(self, other):
        new = self._validate(other)
        if (len(new), len(new[0])) != self.shape():
            raise ValueError(f"size mismatch: {self.shape()} vs {(len(new), len(new[0]))}")
        self.data = new
        return self

    def __eq__(self, other):
        if isinstance(other, matrix):
            return self.data == other.data
        return NotImplemented

    def __add__(self, other):
        if isinstance(other, matrix):
            return self.add(other)
        return NotImplemented

    def __sub__(self, other):
        if isinstance(other, matrix):
            return self.sub(other)
        return NotImplemented

    def __mul__(self, other):
        if isinstance(other, matrix):
            return self.mat_mult(other)
        if isinstance(other, Number) and not isinstance(other, bool):
            return self.scalarmul(other)
        return NotImplemented

    def __rmul__(self, other):
        if isinstance(other, Number) and not isinstance(other, bool):
            return self.scalarmul(other)
        return NotImplemented

    def __repr__(self):
        return "matrix(" + repr(self.data) + ")"


def constant(n, m, c):
    if isinstance(c, bool) or not isinstance(c, Number):
        raise TypeError("c must be a real number")
    M = matrix(n, m)
    M.data = [[float(c)] * m for _ in range(n)]
    return M


def zeros(n, m):
    return constant(n, m, 0)


def ones(n, m):
    return constant(n, m, 1)


def eye(n):
    M = matrix(n, n)
    M.data = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    return M


def raises(exc, f, *a):
    try:
        f(*a)
    except exc:
        return True
    except Exception as e:
        print("  wrong exception:", type(e).__name__, e)
        return False
    return False


def test_part1():
    Z = matrix(2, 3)
    assert Z.data == [[0, 0, 0], [0, 0, 0]] and Z.shape() == (2, 3)
    assert raises(ValueError, matrix, 0, 3)
    assert raises(ValueError, matrix, -1, 3)
    assert raises(TypeError, matrix, 2.5, 3)
    Z[0][0] = 7
    assert Z[1][0] == 0
    print("PASS: matrix(n, m) gives independent rows of zeros; bad sizes rejected")

    A = matrix([[1, 2, 3], [4, 5, 6]])
    assert A.shape() == (2, 3)
    assert raises(ValueError, matrix, [[1, 2, 3], [4, 5]])
    assert raises(ValueError, matrix, [[1, 2], [3, 4], [5]])
    assert raises(ValueError, matrix, [])
    assert raises(ValueError, matrix, [[]])
    assert raises(ValueError, matrix, [1, 2, 3])
    assert raises(TypeError, matrix, [[1, "a"], [3, 4]])
    src = [[1, 2], [3, 4]]
    B = matrix(src)
    src[0][0] = 99
    assert B[0][0] == 1
    print("PASS: list-of-lists init validated and copied")

    assert A[0][1] == 2 and A[0, 1] == 2
    A[0][1] = 20
    assert A[0, 1] == 20
    A[1, 0] = 40
    assert A[1][0] == 40
    assert raises(IndexError, lambda: A[5, 0])
    assert raises(IndexError, lambda: A[0, 9])
    print("PASS: M[i][j] and M[i,j] read and write")

    M1 = matrix(2, 2)
    M2 = matrix([[1, 2], [3, 4]])
    M1.assign(M2)
    assert M1 == M2 and M1 is not M2
    M2[0, 0] = 100
    assert M1[0, 0] == 1
    M1[:] = matrix([[5, 6], [7, 8]])
    assert M1.data == [[5, 6], [7, 8]]
    M1.assign([[9, 10], [11, 12]])
    M1[:] = [[0, 1], [2, 3]]
    assert M1.data == [[0, 1], [2, 3]]
    assert raises(ValueError, M1.assign, matrix(3, 3))
    assert raises(ValueError, M1.assign, [[1, 2, 3], [4, 5, 6]])
    assert raises(ValueError, M1.assign, [[1, 2], [3]])
    assert M1.data == [[0, 1], [2, 3]]
    print("PASS: assignment from matrix / list of lists, errors on bad size")


def test_part2():
    A = matrix([[1, 2, 3], [4, 5, 6]])

    s = A.shape()
    assert s == (2, 3) and isinstance(s, tuple)
    assert matrix(5, 1).shape() == (5, 1)
    print("PASS: shape() returns (n, m) tuple")

    T = A.transpose()
    assert T == matrix([[1, 4], [2, 5], [3, 6]])
    assert T.shape() == (3, 2)
    assert T is not A
    assert T.transpose() == A
    T[0, 0] = 99
    assert A[0, 0] == 1
    assert matrix([[1, 2], [3, 4]]).transpose() == matrix([[1, 3], [2, 4]])
    assert matrix([[1, 2, 3]]).transpose().shape() == (3, 1)
    print("PASS: transpose() correct, new instance, double transpose is identity")

    r = A.row(0)
    assert r == matrix([[1, 2, 3]]) and r.shape() == (1, 3)
    assert A.row(1) == matrix([[4, 5, 6]])
    c = A.column(1)
    assert c == matrix([[2], [5]]) and c.shape() == (2, 1)
    assert A.column(2) == matrix([[3], [6]])
    r[0, 0] = -1
    c[0, 0] = -1
    assert A[0, 0] == 1 and A[0, 1] == 2
    assert raises(IndexError, A.row, 2)
    assert raises(IndexError, A.column, 3)
    print("PASS: row(n)/column(n) return correctly shaped independent matrices")

    L = A.to_list()
    assert L == [[1, 2, 3], [4, 5, 6]] and isinstance(L, list)
    L[0][0] = 99
    assert A[0, 0] == 1
    print("PASS: to_list() returns a list of lists copy")

    B = matrix([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])
    blk = B.block(0, 2, 1, 3)
    assert blk == matrix([[2, 3], [6, 7]]) and blk.shape() == (2, 2)
    assert B.block(0, 3, 0, 4) == B
    assert B.block(2, 3, 3, 4) == matrix([[12]])
    blk[0, 0] = -1
    assert B[0, 1] == 2
    assert raises(ValueError, B.block, 2, 2, 0, 1)
    assert raises(ValueError, B.block, 0, 4, 0, 1)
    assert raises(ValueError, B.block, 0, 1, 0, 5)
    assert raises(ValueError, B.block, -1, 1, 0, 1)
    assert raises(ValueError, B.block, 2, 1, 0, 1)
    print("PASS: block(n_0,n_1,m_0,m_1) returns the right submatrix, bounds checked")

    assert B[1, 2] == 7 and B[1][2] == 7
    assert B[0:2] == matrix([[1, 2, 3, 4], [5, 6, 7, 8]])
    assert B[::2] == matrix([[1, 2, 3, 4], [9, 10, 11, 12]])
    assert B[1:, 2:] == matrix([[7, 8], [11, 12]])
    assert B[0:2, 1:3] == B.block(0, 2, 1, 3)
    assert B[:, 1] == matrix([[2], [6], [10]]) and B[:, 1].shape() == (3, 1)
    assert B[1, :] == matrix([[5, 6, 7, 8]]) and B[1, :].shape() == (1, 4)
    assert B[:, :] == B and B[:, :] is not B
    assert B[-1, -2:] == matrix([[11, 12]])
    sl = B[0:2, 0:2]
    sl[0, 0] = -1
    assert B[0, 0] == 1
    assert raises(ValueError, lambda: B[1:1])
    assert raises(ValueError, lambda: B[0:2, 5:])
    assert raises(IndexError, lambda: B[9, 0:2])
    assert raises(IndexError, lambda: B[0, 1, 2])
    print("PASS: __getitem__ supports slices (rows, submatrices, row/column forms), copies")

    print("\nAll tests passed.")


def all_floats(M):
    return all(type(v) is float for r in M.data for v in r)


def test_part3():
    C = constant(2, 3, 5)
    assert C.shape() == (2, 3)
    assert C.data == [[5.0, 5.0, 5.0], [5.0, 5.0, 5.0]]
    assert all_floats(C)
    assert constant(1, 4, -2.5).data == [[-2.5, -2.5, -2.5, -2.5]]
    assert constant(3, 1, 0).shape() == (3, 1)
    C[0, 0] = 9.0
    assert C[1, 0] == 5.0
    assert raises(ValueError, constant, 0, 3, 1)
    assert raises(TypeError, constant, 2.0, 3, 1)
    assert raises(TypeError, constant, 2, 3, "a")
    assert raises(TypeError, constant, 2, 3, True)
    print("PASS: constant(n,m,c) gives n x m floats of value c, rows independent, bad input rejected")

    Z = zeros(3, 2)
    assert Z.shape() == (3, 2) and Z.data == [[0.0, 0.0]] * 3
    assert all_floats(Z)
    O = ones(2, 4)
    assert O.shape() == (2, 4) and O.data == [[1.0] * 4] * 2
    assert all_floats(O)
    assert zeros(1, 1) == matrix([[0.0]]) and ones(1, 1) == matrix([[1.0]])
    assert raises(ValueError, zeros, -1, 2)
    assert raises(ValueError, ones, 2, 0)
    print("PASS: zeros(n,m) and ones(n,m) give float matrices of 0.0 and 1.0")

    I = eye(3)
    assert I.shape() == (3, 3)
    assert I.data == [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
    assert all_floats(I)
    assert eye(1) == matrix([[1.0]])
    assert I.transpose() == I
    assert all(I[i, j] == (1.0 if i == j else 0.0) for i in range(3) for j in range(3))
    big = eye(6)
    assert sum(big[i, i] for i in range(6)) == 6.0 and sum(map(sum, big.data)) == 6.0
    I[0, 0] = 5.0
    assert eye(3)[0, 0] == 1.0
    assert raises(ValueError, eye, 0)
    assert raises(TypeError, eye, 2.5)
    print("PASS: eye(n) is the n x n float identity, new object each call")

    print("\nAll part 3 tests passed.")


def test_part4():
    A = matrix([[1, 2, 3], [4, 5, 6]])
    B = matrix([[7, 8, 9], [10, 11, 12]])
    C = matrix([[1, 2], [3, 4], [5, 6]])

    S = A.scalarmul(2)
    assert S == matrix([[2, 4, 6], [8, 10, 12]]) and S is not A
    assert A.scalarmul(0.5).data == [[0.5, 1.0, 1.5], [2.0, 2.5, 3.0]]
    assert A.scalarmul(-1) == matrix([[-1, -2, -3], [-4, -5, -6]])
    assert A.scalarmul(0) == matrix([[0, 0, 0], [0, 0, 0]])
    assert A.scalarmul(1) == A
    assert A.scalarmul(2).shape() == (2, 3)
    assert A == matrix([[1, 2, 3], [4, 5, 6]])
    assert raises(TypeError, A.scalarmul, "a")
    assert raises(TypeError, A.scalarmul, [1])
    assert raises(TypeError, A.scalarmul, True)
    print("PASS: scalarmul(c) correct, new matrix, original unchanged, bad c rejected")

    assert A.add(B) == matrix([[8, 10, 12], [14, 16, 18]])
    assert A.add(B) == B.add(A)
    assert A.add(zeros(2, 3)) == A
    assert A.add(A) == A.scalarmul(2)
    assert A.add(B).shape() == (2, 3)
    assert raises(ValueError, A.add, C)
    assert raises(ValueError, A.add, matrix(2, 2))
    assert raises(ValueError, A.add, matrix(3, 2))
    assert raises(TypeError, A.add, [[1, 2, 3], [4, 5, 6]])
    assert A == matrix([[1, 2, 3], [4, 5, 6]]) and B == matrix([[7, 8, 9], [10, 11, 12]])
    print("PASS: add(N) correct, shapes checked, operands unchanged")

    assert A.sub(B) == matrix([[-6, -6, -6], [-6, -6, -6]])
    assert A.sub(A) == zeros(2, 3)
    assert A.sub(B).add(B) == A
    assert B.sub(A) == A.sub(B).scalarmul(-1)
    assert raises(ValueError, A.sub, C)
    assert raises(ValueError, A.sub, matrix(1, 3))
    assert raises(TypeError, A.sub, 5)
    print("PASS: sub(N) correct, shapes checked")

    P = A.mat_mult(C)
    assert P == matrix([[22, 28], [49, 64]]) and P.shape() == (2, 2)
    Q = C.mat_mult(A)
    assert Q == matrix([[9, 12, 15], [19, 26, 33], [29, 40, 51]]) and Q.shape() == (3, 3)
    assert P.shape() != Q.shape()
    assert A.mat_mult(eye(3)) == A
    assert eye(2).mat_mult(A) == A
    v = matrix([[1], [0], [1]])
    assert A.mat_mult(v) == matrix([[4], [10]]) and A.mat_mult(v).shape() == (2, 1)
    assert matrix([[1, 2, 3]]).mat_mult(v) == matrix([[4]])
    assert v.mat_mult(matrix([[1, 2]])).shape() == (3, 2)
    assert A.mat_mult(C).mat_mult(A) == A.mat_mult(C.mat_mult(A))
    assert A.mat_mult(C).transpose() == C.transpose().mat_mult(A.transpose())
    assert A.mat_mult(ones(3, 2).add(ones(3, 2))) == A.mat_mult(ones(3, 2)).scalarmul(2)
    assert raises(ValueError, A.mat_mult, B)
    assert raises(ValueError, A.mat_mult, matrix(2, 2))
    assert raises(ValueError, C.mat_mult, C)
    assert raises(TypeError, A.mat_mult, [[1], [2], [3]])
    print("PASS: mat_mult(N) correct shapes/values, identity and algebraic laws, inner dims checked")

    E = A.element_mult(B)
    assert E == matrix([[7, 16, 27], [40, 55, 72]]) and E.shape() == (2, 3)
    assert E == B.element_mult(A)
    assert A.element_mult(ones(2, 3)) == A
    assert A.element_mult(zeros(2, 3)) == zeros(2, 3)
    assert raises(ValueError, A.element_mult, C)
    assert raises(ValueError, A.element_mult, matrix(2, 4))
    assert raises(TypeError, A.element_mult, 2)
    print("PASS: element_mult(N) correct, shapes checked (2x3 vs 3x2 rejected)")

    assert A.equals(matrix([[1, 2, 3], [4, 5, 6]])) is True
    assert A.equals(A) is True
    assert A.equals(B) is False
    assert A.equals(matrix([[1, 2, 3], [4, 5, 7]])) is False
    assert A.equals(C) is False
    assert A.equals(A.transpose()) is False
    assert A.equals(matrix([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])) is True
    assert A.equals([[1, 2, 3], [4, 5, 6]]) is False
    assert A.equals(None) is False
    print("PASS: equals(N) returns True/False, different shapes are simply False")

    print("\nAll part 4 tests passed.")


def test_part5():
    A = matrix([[1, 2, 3], [4, 5, 6]])
    B = matrix([[7, 8, 9], [10, 11, 12]])
    C = matrix([[1, 2], [3, 4], [5, 6]])

    assert 2 * A == A.scalarmul(2) == matrix([[2, 4, 6], [8, 10, 12]])
    assert A * 2 == 2 * A
    assert 2.5 * A == A * 2.5 == A.scalarmul(2.5)
    assert 0 * A == zeros(2, 3)
    assert (-1) * A == matrix([[-1, -2, -3], [-4, -5, -6]])
    assert 1 * A == A and 1 * A is not A
    assert 2 * A * 3 == A.scalarmul(6)
    assert 2 * (A + B) == 2 * A + 2 * B
    assert (2 * A).shape() == (A * 2).shape() == (2, 3)
    assert raises(TypeError, lambda: "x" * A)
    assert raises(TypeError, lambda: A * "x")
    assert raises(TypeError, lambda: A * None)
    assert raises(TypeError, lambda: True * A)
    assert raises(TypeError, lambda: A * True)
    assert A == matrix([[1, 2, 3], [4, 5, 6]])
    print("PASS: 2*M and M*2 scale every entry; bad scalars raise TypeError")

    assert A + B == A.add(B) == matrix([[8, 10, 12], [14, 16, 18]])
    assert A + B == B + A
    assert A + A == 2 * A
    assert A + B + A == matrix([[9, 12, 15], [18, 21, 24]])
    assert A + zeros(2, 3) == A
    assert raises(ValueError, lambda: A + C)
    assert raises(ValueError, lambda: A + matrix(2, 2))
    assert raises(TypeError, lambda: A + 1)
    assert raises(TypeError, lambda: 1 + A)
    assert raises(TypeError, lambda: A + [[1, 2, 3], [4, 5, 6]])
    A2 = A
    A2 += B
    assert A2 == matrix([[8, 10, 12], [14, 16, 18]])
    assert A == matrix([[1, 2, 3], [4, 5, 6]])
    print("PASS: M+N works, shape-checked, operands unchanged")

    assert A - B == A.sub(B) == matrix([[-6, -6, -6], [-6, -6, -6]])
    assert A - A == zeros(2, 3)
    assert A + B - B == A
    assert B - A == (-1) * (A - B)
    assert raises(ValueError, lambda: A - C)
    assert raises(TypeError, lambda: A - 1)
    assert raises(TypeError, lambda: 1 - A)
    print("PASS: M-N works, shape-checked")

    assert A * C == A.mat_mult(C) == matrix([[22, 28], [49, 64]])
    assert C * A == C.mat_mult(A) and (C * A).shape() == (3, 3)
    assert (A * C).shape() == (2, 2)
    assert A * eye(3) == A
    assert eye(2) * A == A
    assert A * matrix([[1], [0], [1]]) == matrix([[4], [10]])
    assert (A * C) * A == A * (C * A)
    assert (A + B) * C == A * C + B * C
    assert (A * C).transpose() == C.transpose() * A.transpose()
    assert 2 * (A * C) == (2 * A) * C == A * (2 * C)
    assert not (A * C == C * A)
    assert raises(ValueError, lambda: A * B)
    assert raises(ValueError, lambda: C * C)
    assert raises(ValueError, lambda: A * matrix(2, 2))
    print("PASS: M*N is the matrix product, inner dimensions checked, algebraic laws hold")

    assert (A == matrix([[1, 2, 3], [4, 5, 6]])) is True
    assert (A == A) is True
    assert (A == B) is False
    assert (A == C) is False
    assert (A == A.transpose()) is False
    assert (A != B) is True
    assert (A != matrix([[1, 2, 3], [4, 5, 6]])) is False
    assert (A == matrix([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])) is True
    assert (A == [[1, 2, 3], [4, 5, 6]]) is False
    assert (A == None) is False
    assert (A == 5) is False
    print("PASS: M==N and M!=N return True/False, different shapes/types are just False")

    print("\nAll part 5 tests passed.")


if __name__ == "__main__":
    test_part1()
    test_part2()
    test_part3()
    test_part4()
    test_part5()

PASS: matrix(n, m) gives independent rows of zeros; bad sizes rejected
PASS: list-of-lists init validated and copied
PASS: M[i][j] and M[i,j] read and write
PASS: assignment from matrix / list of lists, errors on bad size
PASS: shape() returns (n, m) tuple
PASS: transpose() correct, new instance, double transpose is identity
PASS: row(n)/column(n) return correctly shaped independent matrices
PASS: to_list() returns a list of lists copy
PASS: block(n_0,n_1,m_0,m_1) returns the right submatrix, bounds checked
PASS: __getitem__ supports slices (rows, submatrices, row/column forms), copies

All tests passed.
PASS: constant(n,m,c) gives n x m floats of value c, rows independent, bad input rejected
PASS: zeros(n,m) and ones(n,m) give float matrices of 0.0 and 1.0
PASS: eye(n) is the n x n float identity, new object each call

All part 3 tests passed.
PASS: scalarmul(c) correct, new matrix, original unchanged, bad c rejected
PASS: add(N) correct, shapes checked, operands unchanged
PASS: sub(N

In [7]:
from numbers import Number

class matrix:
    def __init__(self, *args):
        if len(args) == 2:
            n, m = args
            if not all(isinstance(v, int) and not isinstance(v, bool) for v in (n, m)):
                raise TypeError("n and m must be integers")
            if n <= 0 or m <= 0:
                raise ValueError("n and m must be positive")
            self.data = [[0] * m for _ in range(n)]
        elif len(args) == 1:
            self.data = self._validate(args[0])
        else:
            raise TypeError("matrix() takes (n, m) or a list of lists")

    @staticmethod
    def _validate(rows):
        if isinstance(rows, matrix):
            rows = rows.data
        if not isinstance(rows, list) or len(rows) == 0:
            raise ValueError("expected a non-empty list of lists")
        if not all(isinstance(r, list) for r in rows):
            raise ValueError("every row must be a list")
        width = len(rows[0])
        if width == 0:
            raise ValueError("rows must not be empty")
        for k, r in enumerate(rows):
            if len(r) != width:
                raise ValueError(f"row {k} has {len(r)} columns, expected {width}")
            for v in r:
                if isinstance(v, bool) or not isinstance(v, Number):
                    raise TypeError(f"non-numeric entry {v!r} in row {k}")
        return [list(r) for r in rows]

    def shape(self):
        return (len(self.data), len(self.data[0]))

    def transpose(self):
        return matrix([list(col) for col in zip(*self.data)])

    def row(self, n):
        return matrix([list(self.data[n])])

    def column(self, n):
        return matrix([[r[n]] for r in self.data])

    def to_list(self):
        return [list(r) for r in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        n, m = self.shape()
        if not (0 <= n_0 < n_1 <= n and 0 <= m_0 < m_1 <= m):
            raise ValueError(
                f"block bounds ({n_0},{n_1},{m_0},{m_1}) invalid for shape {(n, m)}"
            )
        return matrix([r[m_0:m_1] for r in self.data[n_0:n_1]])

    @staticmethod
    def _require_matrix(N):
        if not isinstance(N, matrix):
            raise TypeError(f"expected a matrix, got {type(N).__name__}")

    def _require_same_shape(self, N, op):
        self._require_matrix(N)
        if self.shape() != N.shape():
            raise ValueError(f"{op}: shape mismatch {self.shape()} vs {N.shape()}")

    def scalarmul(self, c):
        if isinstance(c, bool) or not isinstance(c, Number):
            raise TypeError("c must be a number")
        return matrix([[c * v for v in r] for r in self.data])

    def add(self, N):
        self._require_same_shape(N, "add")
        return matrix([[a + b for a, b in zip(r, s)] for r, s in zip(self.data, N.data)])

    def sub(self, N):
        self._require_same_shape(N, "sub")
        return matrix([[a - b for a, b in zip(r, s)] for r, s in zip(self.data, N.data)])

    def mat_mult(self, N):
        self._require_matrix(N)
        if self.shape()[1] != N.shape()[0]:
            raise ValueError(
                f"mat_mult: inner dimensions differ, {self.shape()} times {N.shape()}"
            )
        cols = list(zip(*N.data))
        return matrix([[sum(a * b for a, b in zip(r, col)) for col in cols] for r in self.data])

    def element_mult(self, N):
        self._require_same_shape(N, "element_mult")
        return matrix([[a * b for a, b in zip(r, s)] for r, s in zip(self.data, N.data)])

    def equals(self, N):
        return isinstance(N, matrix) and self.data == N.data

    def __getitem__(self, key):
        if isinstance(key, tuple):
            if len(key) != 2:
                raise IndexError("matrix indices must be M[i], M[i,j] or slices thereof")
            i, j = key
            if not isinstance(i, slice) and not isinstance(j, slice):
                return self.data[i][j]
            rows = self.data[i] if isinstance(i, slice) else [self.data[i]]
            if isinstance(j, slice):
                return matrix([r[j] for r in rows])
            return matrix([[r[j]] for r in rows])
        if isinstance(key, slice):
            return matrix(self.data[key])
        return self.data[key]

    def __setitem__(self, key, value):
        if isinstance(key, tuple):
            i, j = key
            self.data[i][j] = value
        elif isinstance(key, slice) and key == slice(None):
            self.assign(value)
        else:
            if not isinstance(value, list) or len(value) != self.shape()[1]:
                raise ValueError("row has wrong length")
            self.data[key] = list(value)

    def assign(self, other):
        new = self._validate(other)
        if (len(new), len(new[0])) != self.shape():
            raise ValueError(f"size mismatch: {self.shape()} vs {(len(new), len(new[0]))}")
        self.data = new
        return self

    def __eq__(self, other):
        if isinstance(other, matrix):
            return self.data == other.data
        return NotImplemented

    def __add__(self, other):
        if isinstance(other, matrix):
            return self.add(other)
        return NotImplemented

    def __sub__(self, other):
        if isinstance(other, matrix):
            return self.sub(other)
        return NotImplemented

    def __mul__(self, other):
        if isinstance(other, matrix):
            return self.mat_mult(other)
        if isinstance(other, Number) and not isinstance(other, bool):
            return self.scalarmul(other)
        return NotImplemented

    def __rmul__(self, other):
        if isinstance(other, Number) and not isinstance(other, bool):
            return self.scalarmul(other)
        return NotImplemented

    def __repr__(self):
        return "matrix(" + repr(self.data) + ")"


def constant(n, m, c):
    if isinstance(c, bool) or not isinstance(c, Number):
        raise TypeError("c must be a real number")
    M = matrix(n, m)
    M.data = [[float(c)] * m for _ in range(n)]
    return M


def zeros(n, m):
    return constant(n, m, 0)


def ones(n, m):
    return constant(n, m, 1)


def eye(n):
    M = matrix(n, n)
    M.data = [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    return M


def raises(exc, f, *a):
    try:
        f(*a)
    except exc:
        return True
    except Exception as e:
        print("  wrong exception:", type(e).__name__, e)
        return False
    return False


def test_part1():
    Z = matrix(2, 3)
    assert Z.data == [[0, 0, 0], [0, 0, 0]] and Z.shape() == (2, 3)
    assert raises(ValueError, matrix, 0, 3)
    assert raises(ValueError, matrix, -1, 3)
    assert raises(TypeError, matrix, 2.5, 3)
    Z[0][0] = 7
    assert Z[1][0] == 0
    print("PASS: matrix(n, m) gives independent rows of zeros; bad sizes rejected")

    A = matrix([[1, 2, 3], [4, 5, 6]])
    assert A.shape() == (2, 3)
    assert raises(ValueError, matrix, [[1, 2, 3], [4, 5]])
    assert raises(ValueError, matrix, [[1, 2], [3, 4], [5]])
    assert raises(ValueError, matrix, [])
    assert raises(ValueError, matrix, [[]])
    assert raises(ValueError, matrix, [1, 2, 3])
    assert raises(TypeError, matrix, [[1, "a"], [3, 4]])
    src = [[1, 2], [3, 4]]
    B = matrix(src)
    src[0][0] = 99
    assert B[0][0] == 1
    print("PASS: list-of-lists init validated and copied")

    assert A[0][1] == 2 and A[0, 1] == 2
    A[0][1] = 20
    assert A[0, 1] == 20
    A[1, 0] = 40
    assert A[1][0] == 40
    assert raises(IndexError, lambda: A[5, 0])
    assert raises(IndexError, lambda: A[0, 9])
    print("PASS: M[i][j] and M[i,j] read and write")

    M1 = matrix(2, 2)
    M2 = matrix([[1, 2], [3, 4]])
    M1.assign(M2)
    assert M1 == M2 and M1 is not M2
    M2[0, 0] = 100
    assert M1[0, 0] == 1
    M1[:] = matrix([[5, 6], [7, 8]])
    assert M1.data == [[5, 6], [7, 8]]
    M1.assign([[9, 10], [11, 12]])
    M1[:] = [[0, 1], [2, 3]]
    assert M1.data == [[0, 1], [2, 3]]
    assert raises(ValueError, M1.assign, matrix(3, 3))
    assert raises(ValueError, M1.assign, [[1, 2, 3], [4, 5, 6]])
    assert raises(ValueError, M1.assign, [[1, 2], [3]])
    assert M1.data == [[0, 1], [2, 3]]
    print("PASS: assignment from matrix / list of lists, errors on bad size")


def test_part2():
    A = matrix([[1, 2, 3], [4, 5, 6]])

    s = A.shape()
    assert s == (2, 3) and isinstance(s, tuple)
    assert matrix(5, 1).shape() == (5, 1)
    print("PASS: shape() returns (n, m) tuple")

    T = A.transpose()
    assert T == matrix([[1, 4], [2, 5], [3, 6]])
    assert T.shape() == (3, 2)
    assert T is not A
    assert T.transpose() == A
    T[0, 0] = 99
    assert A[0, 0] == 1
    assert matrix([[1, 2], [3, 4]]).transpose() == matrix([[1, 3], [2, 4]])
    assert matrix([[1, 2, 3]]).transpose().shape() == (3, 1)
    print("PASS: transpose() correct, new instance, double transpose is identity")

    r = A.row(0)
    assert r == matrix([[1, 2, 3]]) and r.shape() == (1, 3)
    assert A.row(1) == matrix([[4, 5, 6]])
    c = A.column(1)
    assert c == matrix([[2], [5]]) and c.shape() == (2, 1)
    assert A.column(2) == matrix([[3], [6]])
    r[0, 0] = -1
    c[0, 0] = -1
    assert A[0, 0] == 1 and A[0, 1] == 2
    assert raises(IndexError, A.row, 2)
    assert raises(IndexError, A.column, 3)
    print("PASS: row(n)/column(n) return correctly shaped independent matrices")

    L = A.to_list()
    assert L == [[1, 2, 3], [4, 5, 6]] and isinstance(L, list)
    L[0][0] = 99
    assert A[0, 0] == 1
    print("PASS: to_list() returns a list of lists copy")

    B = matrix([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]])
    blk = B.block(0, 2, 1, 3)
    assert blk == matrix([[2, 3], [6, 7]]) and blk.shape() == (2, 2)
    assert B.block(0, 3, 0, 4) == B
    assert B.block(2, 3, 3, 4) == matrix([[12]])
    blk[0, 0] = -1
    assert B[0, 1] == 2
    assert raises(ValueError, B.block, 2, 2, 0, 1)
    assert raises(ValueError, B.block, 0, 4, 0, 1)
    assert raises(ValueError, B.block, 0, 1, 0, 5)
    assert raises(ValueError, B.block, -1, 1, 0, 1)
    assert raises(ValueError, B.block, 2, 1, 0, 1)
    print("PASS: block(n_0,n_1,m_0,m_1) returns the right submatrix, bounds checked")

    assert B[1, 2] == 7 and B[1][2] == 7
    assert B[0:2] == matrix([[1, 2, 3, 4], [5, 6, 7, 8]])
    assert B[::2] == matrix([[1, 2, 3, 4], [9, 10, 11, 12]])
    assert B[1:, 2:] == matrix([[7, 8], [11, 12]])
    assert B[0:2, 1:3] == B.block(0, 2, 1, 3)
    assert B[:, 1] == matrix([[2], [6], [10]]) and B[:, 1].shape() == (3, 1)
    assert B[1, :] == matrix([[5, 6, 7, 8]]) and B[1, :].shape() == (1, 4)
    assert B[:, :] == B and B[:, :] is not B
    assert B[-1, -2:] == matrix([[11, 12]])
    sl = B[0:2, 0:2]
    sl[0, 0] = -1
    assert B[0, 0] == 1
    assert raises(ValueError, lambda: B[1:1])
    assert raises(ValueError, lambda: B[0:2, 5:])
    assert raises(IndexError, lambda: B[9, 0:2])
    assert raises(IndexError, lambda: B[0, 1, 2])
    print("PASS: __getitem__ supports slices (rows, submatrices, row/column forms), copies")

    print("\nAll tests passed.")


def all_floats(M):
    return all(type(v) is float for r in M.data for v in r)


def test_part3():
    C = constant(2, 3, 5)
    assert C.shape() == (2, 3)
    assert C.data == [[5.0, 5.0, 5.0], [5.0, 5.0, 5.0]]
    assert all_floats(C)
    assert constant(1, 4, -2.5).data == [[-2.5, -2.5, -2.5, -2.5]]
    assert constant(3, 1, 0).shape() == (3, 1)
    C[0, 0] = 9.0
    assert C[1, 0] == 5.0
    assert raises(ValueError, constant, 0, 3, 1)
    assert raises(TypeError, constant, 2.0, 3, 1)
    assert raises(TypeError, constant, 2, 3, "a")
    assert raises(TypeError, constant, 2, 3, True)
    print("PASS: constant(n,m,c) gives n x m floats of value c, rows independent, bad input rejected")

    Z = zeros(3, 2)
    assert Z.shape() == (3, 2) and Z.data == [[0.0, 0.0]] * 3
    assert all_floats(Z)
    O = ones(2, 4)
    assert O.shape() == (2, 4) and O.data == [[1.0] * 4] * 2
    assert all_floats(O)
    assert zeros(1, 1) == matrix([[0.0]]) and ones(1, 1) == matrix([[1.0]])
    assert raises(ValueError, zeros, -1, 2)
    assert raises(ValueError, ones, 2, 0)
    print("PASS: zeros(n,m) and ones(n,m) give float matrices of 0.0 and 1.0")

    I = eye(3)
    assert I.shape() == (3, 3)
    assert I.data == [[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]]
    assert all_floats(I)
    assert eye(1) == matrix([[1.0]])
    assert I.transpose() == I
    assert all(I[i, j] == (1.0 if i == j else 0.0) for i in range(3) for j in range(3))
    big = eye(6)
    assert sum(big[i, i] for i in range(6)) == 6.0 and sum(map(sum, big.data)) == 6.0
    I[0, 0] = 5.0
    assert eye(3)[0, 0] == 1.0
    assert raises(ValueError, eye, 0)
    assert raises(TypeError, eye, 2.5)
    print("PASS: eye(n) is the n x n float identity, new object each call")

    print("\nAll part 3 tests passed.")


def test_part4():
    A = matrix([[1, 2, 3], [4, 5, 6]])
    B = matrix([[7, 8, 9], [10, 11, 12]])
    C = matrix([[1, 2], [3, 4], [5, 6]])

    S = A.scalarmul(2)
    assert S == matrix([[2, 4, 6], [8, 10, 12]]) and S is not A
    assert A.scalarmul(0.5).data == [[0.5, 1.0, 1.5], [2.0, 2.5, 3.0]]
    assert A.scalarmul(-1) == matrix([[-1, -2, -3], [-4, -5, -6]])
    assert A.scalarmul(0) == matrix([[0, 0, 0], [0, 0, 0]])
    assert A.scalarmul(1) == A
    assert A.scalarmul(2).shape() == (2, 3)
    assert A == matrix([[1, 2, 3], [4, 5, 6]])
    assert raises(TypeError, A.scalarmul, "a")
    assert raises(TypeError, A.scalarmul, [1])
    assert raises(TypeError, A.scalarmul, True)
    print("PASS: scalarmul(c) correct, new matrix, original unchanged, bad c rejected")

    assert A.add(B) == matrix([[8, 10, 12], [14, 16, 18]])
    assert A.add(B) == B.add(A)
    assert A.add(zeros(2, 3)) == A
    assert A.add(A) == A.scalarmul(2)
    assert A.add(B).shape() == (2, 3)
    assert raises(ValueError, A.add, C)
    assert raises(ValueError, A.add, matrix(2, 2))
    assert raises(ValueError, A.add, matrix(3, 2))
    assert raises(TypeError, A.add, [[1, 2, 3], [4, 5, 6]])
    assert A == matrix([[1, 2, 3], [4, 5, 6]]) and B == matrix([[7, 8, 9], [10, 11, 12]])
    print("PASS: add(N) correct, shapes checked, operands unchanged")

    assert A.sub(B) == matrix([[-6, -6, -6], [-6, -6, -6]])
    assert A.sub(A) == zeros(2, 3)
    assert A.sub(B).add(B) == A
    assert B.sub(A) == A.sub(B).scalarmul(-1)
    assert raises(ValueError, A.sub, C)
    assert raises(ValueError, A.sub, matrix(1, 3))
    assert raises(TypeError, A.sub, 5)
    print("PASS: sub(N) correct, shapes checked")

    P = A.mat_mult(C)
    assert P == matrix([[22, 28], [49, 64]]) and P.shape() == (2, 2)
    Q = C.mat_mult(A)
    assert Q == matrix([[9, 12, 15], [19, 26, 33], [29, 40, 51]]) and Q.shape() == (3, 3)
    assert P.shape() != Q.shape()
    assert A.mat_mult(eye(3)) == A
    assert eye(2).mat_mult(A) == A
    v = matrix([[1], [0], [1]])
    assert A.mat_mult(v) == matrix([[4], [10]]) and A.mat_mult(v).shape() == (2, 1)
    assert matrix([[1, 2, 3]]).mat_mult(v) == matrix([[4]])
    assert v.mat_mult(matrix([[1, 2]])).shape() == (3, 2)
    assert A.mat_mult(C).mat_mult(A) == A.mat_mult(C.mat_mult(A))
    assert A.mat_mult(C).transpose() == C.transpose().mat_mult(A.transpose())
    assert A.mat_mult(ones(3, 2).add(ones(3, 2))) == A.mat_mult(ones(3, 2)).scalarmul(2)
    assert raises(ValueError, A.mat_mult, B)
    assert raises(ValueError, A.mat_mult, matrix(2, 2))
    assert raises(ValueError, C.mat_mult, C)
    assert raises(TypeError, A.mat_mult, [[1], [2], [3]])
    print("PASS: mat_mult(N) correct shapes/values, identity and algebraic laws, inner dims checked")

    E = A.element_mult(B)
    assert E == matrix([[7, 16, 27], [40, 55, 72]]) and E.shape() == (2, 3)
    assert E == B.element_mult(A)
    assert A.element_mult(ones(2, 3)) == A
    assert A.element_mult(zeros(2, 3)) == zeros(2, 3)
    assert raises(ValueError, A.element_mult, C)
    assert raises(ValueError, A.element_mult, matrix(2, 4))
    assert raises(TypeError, A.element_mult, 2)
    print("PASS: element_mult(N) correct, shapes checked (2x3 vs 3x2 rejected)")

    assert A.equals(matrix([[1, 2, 3], [4, 5, 6]])) is True
    assert A.equals(A) is True
    assert A.equals(B) is False
    assert A.equals(matrix([[1, 2, 3], [4, 5, 7]])) is False
    assert A.equals(C) is False
    assert A.equals(A.transpose()) is False
    assert A.equals(matrix([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])) is True
    assert A.equals([[1, 2, 3], [4, 5, 6]]) is False
    assert A.equals(None) is False
    print("PASS: equals(N) returns True/False, different shapes are simply False")

    print("\nAll part 4 tests passed.")


def test_part5():
    A = matrix([[1, 2, 3], [4, 5, 6]])
    B = matrix([[7, 8, 9], [10, 11, 12]])
    C = matrix([[1, 2], [3, 4], [5, 6]])

    assert 2 * A == A.scalarmul(2) == matrix([[2, 4, 6], [8, 10, 12]])
    assert A * 2 == 2 * A
    assert 2.5 * A == A * 2.5 == A.scalarmul(2.5)
    assert 0 * A == zeros(2, 3)
    assert (-1) * A == matrix([[-1, -2, -3], [-4, -5, -6]])
    assert 1 * A == A and 1 * A is not A
    assert 2 * A * 3 == A.scalarmul(6)
    assert 2 * (A + B) == 2 * A + 2 * B
    assert (2 * A).shape() == (A * 2).shape() == (2, 3)
    assert raises(TypeError, lambda: "x" * A)
    assert raises(TypeError, lambda: A * "x")
    assert raises(TypeError, lambda: A * None)
    assert raises(TypeError, lambda: True * A)
    assert raises(TypeError, lambda: A * True)
    assert A == matrix([[1, 2, 3], [4, 5, 6]])
    print("PASS: 2*M and M*2 scale every entry; bad scalars raise TypeError")

    assert A + B == A.add(B) == matrix([[8, 10, 12], [14, 16, 18]])
    assert A + B == B + A
    assert A + A == 2 * A
    assert A + B + A == matrix([[9, 12, 15], [18, 21, 24]])
    assert A + zeros(2, 3) == A
    assert raises(ValueError, lambda: A + C)
    assert raises(ValueError, lambda: A + matrix(2, 2))
    assert raises(TypeError, lambda: A + 1)
    assert raises(TypeError, lambda: 1 + A)
    assert raises(TypeError, lambda: A + [[1, 2, 3], [4, 5, 6]])
    A2 = A
    A2 += B
    assert A2 == matrix([[8, 10, 12], [14, 16, 18]])
    assert A == matrix([[1, 2, 3], [4, 5, 6]])
    print("PASS: M+N works, shape-checked, operands unchanged")

    assert A - B == A.sub(B) == matrix([[-6, -6, -6], [-6, -6, -6]])
    assert A - A == zeros(2, 3)
    assert A + B - B == A
    assert B - A == (-1) * (A - B)
    assert raises(ValueError, lambda: A - C)
    assert raises(TypeError, lambda: A - 1)
    assert raises(TypeError, lambda: 1 - A)
    print("PASS: M-N works, shape-checked")

    assert A * C == A.mat_mult(C) == matrix([[22, 28], [49, 64]])
    assert C * A == C.mat_mult(A) and (C * A).shape() == (3, 3)
    assert (A * C).shape() == (2, 2)
    assert A * eye(3) == A
    assert eye(2) * A == A
    assert A * matrix([[1], [0], [1]]) == matrix([[4], [10]])
    assert (A * C) * A == A * (C * A)
    assert (A + B) * C == A * C + B * C
    assert (A * C).transpose() == C.transpose() * A.transpose()
    assert 2 * (A * C) == (2 * A) * C == A * (2 * C)
    assert not (A * C == C * A)
    assert raises(ValueError, lambda: A * B)
    assert raises(ValueError, lambda: C * C)
    assert raises(ValueError, lambda: A * matrix(2, 2))
    print("PASS: M*N is the matrix product, inner dimensions checked, algebraic laws hold")

    assert (A == matrix([[1, 2, 3], [4, 5, 6]])) is True
    assert (A == A) is True
    assert (A == B) is False
    assert (A == C) is False
    assert (A == A.transpose()) is False
    assert (A != B) is True
    assert (A != matrix([[1, 2, 3], [4, 5, 6]])) is False
    assert (A == matrix([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])) is True
    assert (A == [[1, 2, 3], [4, 5, 6]]) is False
    assert (A == None) is False
    assert (A == 5) is False
    print("PASS: M==N and M!=N return True/False, different shapes/types are just False")

    print("\nAll part 5 tests passed.")


def show(label, M):
    rows = [" ".join(f"{v:>6}" for v in r) for r in M.to_list()]
    pad = " " * (len(label) + 3)
    print(f"{label} = [{rows[0]} ]")
    for r in rows[1:]:
        print(f"{pad}[{r} ]")


A = matrix([[1, 2], [3, 4]])
B = matrix([[0, 1], [-1, 2]])
C = matrix([[2, 0], [1, 3]])
I = eye(2)

print("Example matrices (all 2 x 2)")
show("A", A)
show("B", B)
show("C", C)
show("I", I)

print("\n1. Associativity: (AB)C = A(BC)")
AB = A * B
BC = B * C
left = AB * C
right = A * BC
show("AB", AB)
show("(AB)C", left)
show("BC", BC)
show("A(BC)", right)
print("(AB)C == A(BC):", left == right)
assert left == right

print("\n2. Distributivity: A(B+C) = AB + AC")
B_plus_C = B + C
left = A * B_plus_C
AC = A * C
right = AB + AC
show("B+C", B_plus_C)
show("A(B+C)", left)
show("AB", AB)
show("AC", AC)
show("AB+AC", right)
print("A(B+C) == AB+AC:", left == right)
assert left == right

print("\n3. Non-commutativity: AB is not BA")
BA = B * A
show("AB", AB)
show("BA", BA)
print("AB == BA:", AB == BA)
assert AB != BA

print("\n4. Identity: AI = A (and IA = A)")
AI = A * I
IA = I * A
show("AI", AI)
show("IA", IA)
print("AI == A:", AI == A)
print("IA == A:", IA == A)
assert AI == A and IA == A

print("\nAll four properties demonstrated.")

Example matrices (all 2 x 2)
A = [     1      2 ]
    [     3      4 ]
B = [     0      1 ]
    [    -1      2 ]
C = [     2      0 ]
    [     1      3 ]
I = [   1.0    0.0 ]
    [   0.0    1.0 ]

1. Associativity: (AB)C = A(BC)
AB = [    -2      5 ]
     [    -4     11 ]
(AB)C = [     1     15 ]
        [     3     33 ]
BC = [     1      3 ]
     [     0      6 ]
A(BC) = [     1     15 ]
        [     3     33 ]
(AB)C == A(BC): True

2. Distributivity: A(B+C) = AB + AC
B+C = [     2      1 ]
      [     0      5 ]
A(B+C) = [     2     11 ]
         [     6     23 ]
AB = [    -2      5 ]
     [    -4     11 ]
AC = [     4      6 ]
     [    10     12 ]
AB+AC = [     2     11 ]
        [     6     23 ]
A(B+C) == AB+AC: True

3. Non-commutativity: AB is not BA
AB = [    -2      5 ]
     [    -4     11 ]
BA = [     3      4 ]
     [     5      6 ]
AB == BA: False

4. Identity: AI = A (and IA = A)
AI = [   1.0    2.0 ]
     [   3.0    4.0 ]
IA = [   1.0    2.0 ]
     [   3.0    4.0 ]
AI =